# Material Usability Maximizer MVP - Interactive Demo

このノートブックでは、ベイズ最適化（BoTorch + GP）を用いた材料寿命最大化の適応的実験計画システムをインタラクティブに実行します。

## 概要

**目的**: CCFatigueデータセットを用いて、材料の疲労寿命を最大化する製造条件を効率的に探索

**手法**: 
- **t: iteration** - 反復的に実験条件を探索（t=1から開始）
- **ym: deterioration index** - 劣化指標（疲労寿命）の測定
- **Bayes Optimize fm(x,t)** - ベイズ最適化による次候補点の提案
- **Bayes update → t+1** - 観測データによるGPモデルの更新と次反復へ

## セクション1: プロジェクト概要とセットアップ

In [ ]:
# 必要なライブラリのインポート
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 自作モジュールのインポート
from src.compute_config import ComputeConfig
from src.data_loader import (
    load_ccfatigue_data,
    prepare_initial_data,
    DatabaseEvaluator,
    get_search_bounds,
    describe_data
)
from src.bayesian_optimizer import BayesianOptimizer
from src.visualization import (
    plot_convergence,
    plot_explored_space,
    plot_gp_predictions,
    plot_compute_performance,
    save_results_summary
)

# プロット設定
%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✓ All modules imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## セクション2: 計算環境の選択と確認

3つの計算モードから選択できます:
- `no_parallel`: CPU単一スレッド（再現性重視）
- `cpu_16cores`: CPU 16コア並列処理
- `gpu_16gb`: GPU加速（16GBメモリ制約対応）
- `auto`: 自動検出（GPU利用可能ならGPU、なければCPU）

In [ ]:
# 計算モードの選択（変更可能）
COMPUTE_MODE = "auto"  # "auto", "no_parallel", "cpu_16cores", "gpu_16gb"

# 計算環境の設定
compute_config = ComputeConfig(
    config_path="config.yaml",
    mode=COMPUTE_MODE
)

# 設定情報の表示
print(compute_config.log_config())

## セクション3: データ読み込みと探索

In [ ]:
# 最適化パラメータの設定
M = 10  # 初期データ点数
T = 20  # ベイズ最適化の反復回数
SEED = 42  # 乱数シード
ACQ_FUNCTION = "EI"  # 獲得関数（"EI" or "LogEI"）

print(f"Optimization parameters:")
print(f"  Initial samples (M): {M}")
print(f"  BO iterations (T): {T}")
print(f"  Random seed: {SEED}")
print(f"  Acquisition function: {ACQ_FUNCTION}")

In [ ]:
# CCFatigueデータの読み込み
X_df, y_df, feature_names, objective_name = load_ccfatigue_data(
    dataset_name="tension_compression",
    design_variables=["stress_max", "stress_ratio", "frequency"],
    objective_variable="cycles_to_failure",
    objective_transform="log10"
)

# データの統計情報
data_info = describe_data(X_df, y_df)
print(f"\nDataset statistics:")
print(f"  Total samples: {data_info['n_samples']}")
print(f"  Features: {data_info['feature_names']}")
print(f"  Objective: {objective_name} (log10 transformed)")

In [ ]:
# データの確認
print("\nFirst few samples:")
display(pd.concat([X_df.head(), y_df.head()], axis=1))

print("\nData summary:")
display(X_df.describe())

In [ ]:
# データの可視化（ペアプロット）
df_combined = pd.concat([X_df, y_df], axis=1)
sns.pairplot(df_combined, diag_kind='kde', corner=True)
plt.suptitle('CCFatigue Data - Pair Plot', y=1.02)
plt.tight_layout()
plt.show()

## セクション4: 初期データの準備

In [ ]:
# 初期データの分割
train_X, train_Y, pool_X, pool_Y = prepare_initial_data(X_df, y_df, M=M, seed=SEED)

print(f"Initial training data: {train_X.shape}")
print(f"Pool data: {pool_X.shape}")
print(f"Best initial y: {train_Y.max().item():.4f}")

In [ ]:
# 全データを結合（データベースとして使用）
X_all = torch.cat([train_X, pool_X], dim=0)
y_all = torch.cat([train_Y, pool_Y], dim=0)

# 探索空間の境界
bounds = get_search_bounds(X_df)

print(f"Search bounds:")
for i, name in enumerate(feature_names):
    print(f"  {name}: [{bounds[0, i]:.2f}, {bounds[1, i]:.2f}]")

In [ ]:
# 仮想実験評価器の初期化
evaluator = DatabaseEvaluator(X_all, y_all, compute_config.device)

print(f"Database evaluator initialized with {len(X_all)} data points")

## セクション5: ベイズ最適化の実行

適応的実験計画のメインループ:
1. GPモデルを現在のデータで学習
2. Expected Improvementで次の候補点を提案
3. データベースで最近傍の実験結果を取得
4. データに追加してベイズ更新 → 次反復へ

In [ ]:
# ベイズ最適化器の初期化
optimizer = BayesianOptimizer(
    bounds=bounds,
    evaluator=evaluator,
    compute_config=compute_config,
    acq_function=ACQ_FUNCTION
)

In [ ]:
# 最適化ループの実行
results = optimizer.optimize_loop(
    train_X=train_X,
    train_Y=train_Y,
    T=T,
    verbose=True
)

## セクション6: 中間結果の可視化

In [ ]:
# 収束曲線
fig = plot_convergence(results["history"])
plt.show()

In [ ]:
# 探索された条件空間
fig = plot_explored_space(
    train_X=results["train_X"],
    train_Y=results["train_Y"],
    feature_names=feature_names
)
plt.show()

In [ ]:
# GP予測分布（1Dスライス）
fig = plot_gp_predictions(
    model=results["final_model"],
    bounds=bounds,
    feature_names=feature_names,
    train_X=results["train_X"],
    train_Y=results["train_Y"],
    device=compute_config.device
)
plt.show()

## セクション7: 計算パフォーマンスの分析

In [ ]:
# 計算パフォーマンスの可視化
fig = plot_compute_performance(
    history=results["history"],
    compute_mode=compute_config.mode
)
plt.show()

In [ ]:
# パフォーマンス統計
exec_times = results["history"]["execution_time"][1:]  # 初期状態を除く

print(f"Performance Statistics ({compute_config.mode} mode):")
print(f"  Total iterations: {T}")
print(f"  Total time: {sum(exec_times):.2f}s")
print(f"  Mean iteration time: {np.mean(exec_times):.2f}s")
print(f"  Std iteration time: {np.std(exec_times):.2f}s")
print(f"  Min iteration time: {np.min(exec_times):.2f}s")
print(f"  Max iteration time: {np.max(exec_times):.2f}s")

## セクション8: 最終結果とまとめ

In [ ]:
# 最良条件の表示
best_x = results["best_x"].numpy()
best_y = results["best_y"]

print("="*70)
print("BEST CONDITIONS FOUND")
print("="*70)
print(f"\nBest objective value (log10 cycles): {best_y:.4f}")
print(f"Estimated cycles to failure: {10**best_y:.2e}")
print(f"\nBest conditions:")
for i, name in enumerate(feature_names):
    print(f"  {name}: {best_x[i]:.4f}")
print("\n" + "="*70)

In [ ]:
# 結果の保存
output_dir = "results_notebook"

save_results_summary(
    results=results,
    feature_names=feature_names,
    output_dir=output_dir,
    compute_mode=compute_config.mode
)

print(f"\nResults saved to {output_dir}/")

In [ ]:
# 最適化履歴の確認
history_df = pd.DataFrame({
    "iteration": results["history"]["iteration"],
    "best_y": results["history"]["best_y"],
    "current_y": results["history"]["current_y"],
    "execution_time_s": results["history"]["execution_time"]
})

print("\nOptimization History (last 10 iterations):")
display(history_df.tail(10))

## まとめ

このノートブックでは、Material Usability Maximizer MVPを使用して、以下を実現しました:

1. **計算環境の柔軟な選択**: 3つの計算モード（no_parallel, cpu_16cores, gpu_16gb）から選択
2. **CCFatigueデータの活用**: 疲労試験データを用いた実証実験
3. **適応的実験計画**: ベイズ最適化による効率的な条件探索
4. **包括的な可視化**: 収束曲線、探索空間、GP予測、計算パフォーマンス
5. **実用的な出力**: 最良条件の発見と詳細な結果保存

### 次のステップ

- パラメータ（M, T, SEED, ACQ_FUNCTION）を変更して再実行
- 異なる計算モードで性能比較
- 実際の実験装置との統合（将来の拡張）
- 多目的最適化への拡張（寿命 + コスト + 製造容易性）

---

**Material Usability Maximizer MVP**  
ベイズ最適化による材料寿命最大化システム